In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EDA") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
df = spark.read.csv(
    "hdfs://localhost:9000/DACK/weatherAUS.csv",
    header=True,
    inferSchema=True,
    nullValue="NA"
)

In [4]:
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- MinTemp: double (nullable = true)
 |-- MaxTemp: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Evaporation: double (nullable = true)
 |-- Sunshine: double (nullable = true)
 |-- WindGustDir: string (nullable = true)
 |-- WindGustSpeed: integer (nullable = true)
 |-- WindDir9am: string (nullable = true)
 |-- WindDir3pm: string (nullable = true)
 |-- WindSpeed9am: integer (nullable = true)
 |-- WindSpeed3pm: integer (nullable = true)
 |-- Humidity9am: integer (nullable = true)
 |-- Humidity3pm: integer (nullable = true)
 |-- Pressure9am: double (nullable = true)
 |-- Pressure3pm: double (nullable = true)
 |-- Cloud9am: integer (nullable = true)
 |-- Cloud3pm: integer (nullable = true)
 |-- Temp9am: double (nullable = true)
 |-- Temp3pm: double (nullable = true)
 |-- RainToday: string (nullable = true)
 |-- RainTomorrow: string (nullable = true)



In [1]:
from pyspark.sql.functions import *
from pyspark.ml.feature import StringIndexer

Xử lý duplicate value

In [9]:
duplicates = df.count() - df.distinct().count()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


Xử lý missing value

In [17]:
missing_df = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

missing_df.show()

+----+--------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+-----+--------------+-----------------+----------------+----------------+---------------+
|Date|Location|MinTemp|MaxTemp|Rainfall|Evaporation|Sunshine|WindGustDir|WindGustSpeed|WindDir9am|WindDir3pm|WindSpeed9am|WindSpeed3pm|Humidity9am|Humidity3pm|Pressure9am|Pressure3pm|Cloud9am|Cloud3pm|Temp9am|Temp3pm|RainToday|RainTomorrow|label|Location_index|WindGustDir_index|WindDir9am_index|WindDir3pm_index|RainToday_index|
+----+--------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+-----+--------------+-----------------+----------------+----------------+---------------+
|   0|    

In [22]:
from pyspark.sql.functions import col
numeric_cols = [
    "MinTemp", "MaxTemp", "Rainfall",
    "WindGustSpeed",
    "WindSpeed9am", "WindSpeed3pm",
    "Humidity9am", "Humidity3pm",
    "Pressure9am", "Pressure3pm",
    "Temp9am", "Temp3pm", "Evaporation", "Sunshine", "CLoud9am", "CLoud3pm",
]

In [23]:
for c in numeric_cols:

    median = df.approxQuantile(
        c,
        [0.5],
        0
    )[0]

    df = df.fillna({
        c: median
    })

In [20]:
categorical_cols = [
    "Location",
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
    "RainToday",
    "RainTomorrow"
]

for c in categorical_cols:

    df = df.fillna({
        c: "Unknown"
    })

In [24]:
missing_df = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

missing_df.show()

+----+--------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+-----+--------------+-----------------+----------------+----------------+---------------+
|Date|Location|MinTemp|MaxTemp|Rainfall|Evaporation|Sunshine|WindGustDir|WindGustSpeed|WindDir9am|WindDir3pm|WindSpeed9am|WindSpeed3pm|Humidity9am|Humidity3pm|Pressure9am|Pressure3pm|Cloud9am|Cloud3pm|Temp9am|Temp3pm|RainToday|RainTomorrow|label|Location_index|WindGustDir_index|WindDir9am_index|WindDir3pm_index|RainToday_index|
+----+--------+-------+-------+--------+-----------+--------+-----------+-------------+----------+----------+------------+------------+-----------+-----------+-----------+-----------+--------+--------+-------+-------+---------+------------+-----+--------------+-----------------+----------------+----------------+---------------+
|   0|    

Encode target variable

In [10]:
label_indexer = StringIndexer(
    inputCol="RainTomorrow",
    outputCol="label"
)

df = label_indexer.fit(df).transform(df)

In [11]:
encode_cols = [
    "Location",
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
    "RainToday"
]

for c in encode_cols:

    indexer = StringIndexer(
        inputCol=c,
        outputCol=c + "_index"
    )

    df = indexer.fit(df).transform(df)

In [12]:
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Location: string (nullable = false)
 |-- MinTemp: double (nullable = false)
 |-- MaxTemp: double (nullable = false)
 |-- Rainfall: double (nullable = false)
 |-- Evaporation: double (nullable = true)
 |-- Sunshine: double (nullable = true)
 |-- WindGustDir: string (nullable = false)
 |-- WindGustSpeed: integer (nullable = true)
 |-- WindDir9am: string (nullable = false)
 |-- WindDir3pm: string (nullable = false)
 |-- WindSpeed9am: integer (nullable = true)
 |-- WindSpeed3pm: integer (nullable = true)
 |-- Humidity9am: integer (nullable = true)
 |-- Humidity3pm: integer (nullable = true)
 |-- Pressure9am: double (nullable = false)
 |-- Pressure3pm: double (nullable = false)
 |-- Cloud9am: integer (nullable = true)
 |-- Cloud3pm: integer (nullable = true)
 |-- Temp9am: double (nullable = false)
 |-- Temp3pm: double (nullable = false)
 |-- RainToday: string (nullable = false)
 |-- RainTomorrow: string (nullable = false)
 |-- label: double (nul

In [25]:
from pyspark.sql.functions import (
    to_timestamp,
    month,
    year,
    col
)
# parse date đúng format
df = df.withColumn(
    "DateParsed",
    to_timestamp(col("Date"), "M/d/yyyy")
)

# extract month/year
df = df.withColumn(
    "Month",
    month(col("DateParsed"))
)

df = df.withColumn(
    "Year",
    year(col("DateParsed"))
)

df.select(
    "Date",
    "DateParsed",
    "Month",
    "Year"
).show(5, truncate=False)

+---------+-------------------+-----+----+
|Date     |DateParsed         |Month|Year|
+---------+-------------------+-----+----+
|12/1/2008|2008-12-01 00:00:00|12   |2008|
|12/2/2008|2008-12-02 00:00:00|12   |2008|
|12/3/2008|2008-12-03 00:00:00|12   |2008|
|12/4/2008|2008-12-04 00:00:00|12   |2008|
|12/5/2008|2008-12-05 00:00:00|12   |2008|
+---------+-------------------+-----+----+
only showing top 5 rows


In [26]:
df.coalesce(1) \
  .write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("weather_clean")